In [ ]:
# Import necessary libraries
import json
import os
from typing import Any, Dict
 
import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [2]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [3]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time information, including the current price of Bitcoin. To find the most up-to-date price, you can check reliable financial news websites, cryptocurrency exchanges, or use financial apps that track cryptocurrency prices. Some popular sources include:

- CoinMarketCap (www.coinmarketcap.com)
- CoinGecko (www.coingecko.com)
- Binance (www.binance.com)
- Kraken (www.kraken.com)

These platforms offer live data on the price of Bitcoin and other cryptocurrencies.


In [4]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "000a2493-9a90-4282-810a-130c9bbe274a",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 14:13:50 GMT",
      "content-type": "application/json",
      "content-length": "701",
      "connection": "keep-alive",
      "x-amzn-requestid": "000a2493-9a90-4282-810a-130c9bbe274a"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time information, including the current price of Bitcoin. To find the most up-to-date price, you can check reliable financial news websites, cryptocurrency exchanges, or use financial apps that track cryptocurrency prices. Some popular sources include:\n\n- CoinMarketCap (www.coinmarketcap.com)\n- CoinGecko (www.coingecko.com)\n- Binance (www.binance.com)\n- Kraken (www.kraken.com)\n\nThese platforms offer live data on the price of Bitcoin and other cryptocurrencies."
        }

In [5]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '65577.61000000'}


In [6]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [7]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 65577.6


In [8]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [9]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "8ed22265-9c6f-4739-aadd-ec47690565a3",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 14:13:54 GMT",
      "content-type": "application/json",
      "content-length": "510",
      "connection": "keep-alive",
      "x-amzn-requestid": "8ed22265-9c6f-4739-aadd-ec47690565a3"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking> The User has asked for the current price of Bitcoin. I need to use the \"get_crypto_price\" tool to get this information. The symbol for Bitcoin on Binance is BTCUSDT. </thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_2WCa8f29JUps8VfwCKWIJf",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
  "stopReason": "tool_use",
  "usage": {
    "inputTokens": 

In [10]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_2WCa8f29JUps8VfwCKWIJf


In [11]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 65582.56


In [12]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "eb466d1c-f222-479f-9ba0-4e995a17b47f",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 14:13:56 GMT",
      "content-type": "application/json",
      "content-length": "478",
      "connection": "keep-alive",
      "x-amzn-requestid": "eb466d1c-f222-479f-9ba0-4e995a17b47f"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking> I have used the \"get_crypto_price\" tool and got the result. The current price of Bitcoin (BTCUSDT) is 65582.56 USDT. </thinking>\n\nThe current price of Bitcoin is 65582.56 USDT. Please note that cryptocurrency prices are highly volatile and can change rapidly."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 547,
    "outputTokens": 77,
    "totalTokens": 624
  },
  "metrics": {
    "latencyMs": 667
  }
}


In [13]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

The current price of Bitcoin is 65582.56 USDT. Please note that cryptocurrency prices are highly volatile and can change rapidly.
